# (실습) 시계열 분석

## 예나 날씨 데이터 관련

**문제 1**

예나(Jena) 도시의 날씨 데이터를 활용하여 LSTM 또는 GRU 모델을 구현한 후
성능을 최대한 끌어올리는 과정을 묘사하라.

- 층의 유닛 개수 및 드랍아웃 비율 조정
- RMSprop 등의 옵티마이저의 학습률 조정 및 다른 옵티마이저 활용
- 순환층 이후에 여러 개의 밀집층 적용
- 시퀀스 길이 조정, 샘플 선택 비율 조정 등 기타 특성 엔지니어링 시도.

**힌트**: `recurrent_dropout` 옵션을 사용할 경우 `unroll=True` 옵션을 함께
활용해야 GPU를 활용할 수 있다.
하나의 시퀀스 길이가 100 이하로 지정되어야 함에 주의하라.


In [1]:
import tensorflow as tf
tf.__version__

'2.13.0'

In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
if 'google.colab' in str(get_ipython()):
    !wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
    !unzip jena_climate_2009_2016.csv.zip
else:
    try:
        import wget, zipfile
    except ModuleNotFoundError:
        !pip install wget

    import wget, zipfile
    wget.download('https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip')
    with zipfile.ZipFile('jena_climate_2009_2016.csv.zip', 'r') as zip_ref:
        zip_ref.extractall('./')

In [4]:
import os
fname = os.path.join("jena_climate_2009_2016.csv")

with open(fname) as f:
    data = f.read()

lines = data.split("\n")

In [5]:
header = lines[0].split(",")
lines = lines[1:]

In [6]:
import numpy as np

temperature = np.zeros((len(lines),))
raw_data = np.zeros((len(lines), len(header) - 1))

for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]

    temperature[i] = values[1]    # i 번째 온도
    raw_data[i, :] = values[:]    # i 번째 데이터

**훈련셋, 검증셋, 테스트셋 지정하기**

각 데이터셋의 크기는 다음과 같다.

- 훈련셋: 전체의 50%
- 검증셋: 전체의 25%
- 테스트셋: 전체의 25%

미래에 대한 예측을 실행하므로 훈련셋, 검증셋, 테스트셋 순으로
보다 오래된 데이터를 사용한다.

In [7]:
num_train_samples = int(0.5 * len(raw_data))     # 전체의 50%
num_val_samples   = int(0.25 * len(raw_data))    # 전체의 25%
num_test_samples  = len(raw_data) - num_train_samples - num_val_samples

print("num_train_samples:\t", num_train_samples)
print("num_val_samples:\t", num_val_samples)
print("num_test_samples:\t", num_test_samples)

num_train_samples:	 210225
num_val_samples:	 105112
num_test_samples:	 105114


**데이터 정규화**

In [8]:
# 훈련셋의 평균
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean

# 훈련셋의 표준편차
std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

**5일 단위 시퀀스 데이터 준비**

앞서 언급한 문제의 해결을 위한 모델을 구현하려면
5일 단위 시퀀스 데이터를 준비해야 하지만
`timeseries_dataset_from_array()` 함수를 활용하면 아주 쉽게 해결된다.

In [24]:
from tensorflow import keras

# 1시간에 하나의 데이터 선택
sampling_rate = 6

# 기존: 120시간 (5일치)
sequence_length = 120  # 3일치 정도로 줄이기

# delay도 그에 맞게 다시 설정
delay = sampling_rate * (sequence_length + 24 - 1)  # 24시간 뒤 예측

# 배치 크기
batch_size = 256

train_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples)

val_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples)

test_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples)

생성된 새로운 데이터셋은 훈련셋의 샘플과 타깃을 함께 배치 단위로 묶여있다.
예를 들어, 훈련셋의 첫째 배치의 모양은 다음과 같다.

- 배치 크기: 256
- 시퀀스 샘플 모양: `(120, 14)`, 즉 14개의 특성을 갖는 날씨 5일치 데이터.

In [25]:
for samples, targets in train_dataset:
    print("샘플 모양:", samples.shape)
    print("타깃 모양:", targets.shape)
    break

샘플 모양: (256, 120, 14)
타깃 모양: (256,)


**순환 드랍아웃 적용**

In [28]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))

x = layers.LSTM(64, return_sequences=True)(inputs)
x = layers.Dropout(0.3)(x)
x = layers.LSTM(32)(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm_final_combo.h5", save_best_only=True),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=4, min_lr=1e-6)
]

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])

history = model.fit(
    train_dataset,
    epochs=50,
    batch_size=128,
    validation_data=val_dataset,
    callbacks=callbacks
)

model = keras.models.load_model("jena_lstm_final_combo.h5")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

Epoch 1/50


2025-04-21 17:45:11.869556: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:45:12.126977: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:45:12.182404: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:45:12.301008: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


  2/819 [..............................] - ETA: 42s - loss: 159.7921 - mae: 10.4464  

2025-04-21 17:45:12.404756: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


819/819 [==============================] - ETA: 0s - loss: 24.2684 - mae: 3.5689

2025-04-21 17:45:36.413570: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:45:36.545458: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:45:36.589546: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


819/819 [==============================] - 31s 37ms/step - loss: 24.2684 - mae: 3.5689 - val_loss: 10.4233 - val_mae: 2.5410 - lr: 0.0010
Epoch 2/50
819/819 [==============================] - 30s 37ms/step - loss: 9.7365 - mae: 2.4305 - val_loss: 10.1222 - val_mae: 2.4990 - lr: 0.0010
Epoch 3/50
819/819 [==============================] - 30s 37ms/step - loss: 7.6692 - mae: 2.1615 - val_loss: 10.9991 - val_mae: 2.6141 - lr: 0.0010
Epoch 4/50
819/819 [==============================] - 29s 36ms/step - loss: 6.2423 - mae: 1.9434 - val_loss: 11.2184 - val_mae: 2.6374 - lr: 0.0010
Epoch 5/50
819/819 [==============================] - 30s 37ms/step - loss: 5.2208 - mae: 1.7726 - val_loss: 12.1511 - val_mae: 2.7525 - lr: 0.0010


2025-04-21 17:47:42.421270: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:47:42.555156: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:47:42.611412: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


405/405 [==============================] - 6s 15ms/step - loss: 11.7112 - mae: 2.7004
Test MAE: 2.70


In [21]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(64, return_sequences=True)(inputs)
x = layers.Dropout(0.25)(x)
x = layers.LSTM(32)(x)
x = layers.Dropout(0.25)(x)

outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

# ReduceLROnPlateau: val_loss가 일정 기간동안 개선되지 않으면 optimizer의 lr을 줄여주는 자동 튜닝 도구

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm_dh.h5", save_best_only=True),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])

history = model.fit(train_dataset,
                    epochs=50,
                    validation_data=val_dataset,
                    callbacks=callbacks)

# 모델 불러오기 (파일명 동일해야 함!)
model = keras.models.load_model("jena_lstm_dh.h5")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

Epoch 1/50


2025-04-21 17:19:45.077146: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:19:45.329249: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:19:45.383644: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:19:45.484221: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


  4/820 [..............................] - ETA: 18s - loss: 131.3303 - mae: 9.5630  

2025-04-21 17:19:45.580043: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


820/820 [==============================] - ETA: 0s - loss: 19.7735 - mae: 3.2516

2025-04-21 17:20:04.203143: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:20:04.335459: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:20:04.376619: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


820/820 [==============================] - 24s 28ms/step - loss: 19.7735 - mae: 3.2516 - val_loss: 10.5943 - val_mae: 2.5704 - lr: 0.0010
Epoch 2/50
820/820 [==============================] - 23s 28ms/step - loss: 8.6303 - mae: 2.2836 - val_loss: 11.1651 - val_mae: 2.6369 - lr: 0.0010
Epoch 3/50
820/820 [==============================] - 23s 28ms/step - loss: 6.4642 - mae: 1.9716 - val_loss: 11.7348 - val_mae: 2.6961 - lr: 0.0010
Epoch 4/50
820/820 [==============================] - 23s 27ms/step - loss: 4.8221 - mae: 1.7025 - val_loss: 12.8887 - val_mae: 2.8268 - lr: 5.0000e-04
Epoch 5/50
820/820 [==============================] - 22s 27ms/step - loss: 4.2478 - mae: 1.5965 - val_loss: 13.1915 - val_mae: 2.8535 - lr: 5.0000e-04
Epoch 6/50
820/820 [==============================] - 22s 27ms/step - loss: 3.6809 - mae: 1.4848 - val_loss: 13.4966 - val_mae: 2.8901 - lr: 2.5000e-04


2025-04-21 17:22:01.747102: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:22:01.878415: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-04-21 17:22:01.922337: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


407/407 [==============================] - 5s 11ms/step - loss: 11.9946 - mae: 2.7012
Test MAE: 2.70


## RNN 아키텍처 관련

**문제 2**

[바흐의 합창 모음집 데이터셋](https://homl.info/bach)을 다운로드해서 압축을 풀면 바흐(Bach)의 382개의 합창곡이 들어 있으며
훈련셋, 검증셋, 테스트셋의 폴더로 구분돼 있다.

- 훈련셋 크기: 229
- 검증셋 크기: 76
- 테스트셋 크기: 77

각각의 합창곡은 100개에서 640개의 타임 스텝으로 구성되며, 하나의 타임 스텝은 네 개의 정수로 지정된 특성을 갖는다.
0을 제외한 정수는 피아노 음표를 가리킨다. 0은 비어 있는 음표를 의미한다.

RNN 또는 CNN 모델을 이용하여 네 개의 음표를 포함하는 다음 스텝을 예측하는 모델을 훈련시켜 보아라.
이때 기존 음표의 시퀀스를 입력값으로 이용한다.

또한 훈련된 모델을 이용하여 한 음씩 새로 생성하는 방식으로 바흐 스타일의 합창곡을 작곡해 보아라.
이를 위해 새로 생성된 네 개의 음표를 새로운 입력값으로 사용하여 또다시 새로운 음표를 예측하는 방식을 활용할 수 있다.

**참고:** [구글의 Coconet 모델](https://magenta.tensorflow.org/coconet)

In [4]:
import pandas as pd
import numpy as np
import os

def load_chorales(path):
    data = []
    for file in sorted(os.listdir(path)):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, file), header=None)
            data.append(df.values)
    return data

train_data = load_chorales('/Users/kimdohyeon/건양대학교병원_바이오헬스/Biomedical_AI_Train/DL/250421_실습/jsb_chorales/train')
val_data = load_chorales('/Users/kimdohyeon/건양대학교병원_바이오헬스/Biomedical_AI_Train/DL/250421_실습/jsb_chorales/valid')
test_data = load_chorales('/Users/kimdohyeon/건양대학교병원_바이오헬스/Biomedical_AI_Train/DL/250421_실습/jsb_chorales/test')

In [5]:
def create_sequences(data, seq_len=16):
    X, y = [], []
    for chorale in data:
        if len(chorale) > seq_len:
            for i in range(len(chorale) - seq_len):
                X.append(chorale[i:i+seq_len])
                y.append(chorale[i+seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data)
X_val, y_val = create_sequences(val_data)
X_test, y_test = create_sequences(test_data)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(128, input_shape=(X_train.shape[1], 4), return_sequences=False),
    Dense(64, activation='relu'),
    Dense(4)  # 4개의 음표 예측 (S, A, T, B)
])

model.compile(loss='mse', optimizer='adam', metrics=['mae'])
model.summary()

2025-04-22 09:56:03.605131: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-04-22 09:56:03.605177: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-04-22 09:56:03.605191: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
2025-04-22 09:56:03.605614: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-22 09:56:03.605974: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 128)               68096     
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dense_1 (Dense)             (None, 4)                 260       
                                                                 
Total params: 76612 (299.27 KB)
Trainable params: 76612 (299.27 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=64,
                    validation_data=(X_val, y_val),
                    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])

NameError: name 'EarlyStopping' is not defined